# Modelo vencedor - blend de 3 LSTMs (espaco de anomalia) sobre reducao PLS defasada

Reproduz de forma **autocontida** (sem depender de nenhum outro arquivo do repositorio)
o modelo que gerou a submissao real ao Kaggle na competicao
[previsao-climatica-de-precipitacao-sobre-a-america-do-sul](https://kaggle.com/competitions/previsao-climatica-de-precipitacao-sobre-a-america-do-sul)
(RMSE real **1.73116**).

Codigo completo do projeto (CV walk-forward de 5 dobras que selecionou esta
configuracao, sweep de hiperparametros, notebook de resultados mais detalhado):
[github.com/mazeeqe/WORCAP-2026](https://github.com/mazeeqe/WORCAP-2026).

## Metodologia

1. **Reducao dimensional**: cada variavel atmosferica passa por PLS (Partial Least
   Squares) supervisionado - o alvo Y e a serie de componentes PCA de `tp` defasada 1
   mes a frente (`pls_lagged`: o PLS busca a parte de cada variavel mais ligada a
   precipitacao *futura*, nao so a de maior variancia espacial), ajustada so ate
   2018-12. `tp` em si e reduzido via PCA comum (nao-supervisionado). Numero de
   componentes escolhido automaticamente (o menor que atinja 90% de variancia, com
   teto de 40 para `tp`/PCA e 15 para cada variavel atmosferica/PLS).
2. **LSTM hindcast/forecast**: encoder LSTM sobre 12 meses de historico + decoder
   condicionado no ultimo mes com dado atmosferico disponivel antes do alvo (sem
   vazamento temporal), no `tp` da origem (congelado, simula a ultima observacao
   real disponivel) e no lag (1 a 24 meses a frente). Preve a **anomalia**
   (`tp - climatologia do mes-alvo`) em vez de `tp` bruto - a sazonalidade deixa de
   ser trabalho da rede.
3. **Blend**: combinacao linear de 3 variantes desse LSTM (dropout/weight_decay
   diferentes; uma delas recebe tambem o indice ONI - El Nino/La Nina - como feature
   extra do decoder), encolhida em direcao a climatologia. Os pesos foram ajustados
   por minimos quadrados numa CV walk-forward de 5 dobras (nao reproduzida aqui, so
   o resultado final e usado). RMSE-CV (leave-one-fold-out): **1.778** mm/dia, contra
   1.828 da climatologia.

Sem correcao de vies ENSO pos-hoc (testada separadamente na CV, piorou o RMSE).

In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import torch
import xarray as xr
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
np.random.seed(0)

## 1. Dados

Se este notebook estiver rodando no Kaggle com a competicao anexada ("Add Input"),
le direto de `/kaggle/input/`. Fora do Kaggle, baixa via `kagglehub` (precisa de
credenciais da API do Kaggle - ver README do projeto).

In [ ]:
COMPETITION = "previsao-climatica-de-precipitacao-sobre-a-america-do-sul"
KAGGLE_INPUT_DIR = f"/kaggle/input/{COMPETITION}"


def load_dataset_files(path):
    """Percorre `path` carregando .csv (pandas) e .nc (xarray) por extensao."""
    dataframes, datasets = {}, {}
    for root, _dirs, files in sorted(os.walk(path)):
        for filename in sorted(files):
            file_path = os.path.join(root, filename)
            key = os.path.splitext(filename)[0]
            lower_name = filename.lower()
            try:
                if lower_name.endswith(".csv"):
                    dataframes[key] = pd.read_csv(file_path)
                elif lower_name.endswith(".nc"):
                    datasets[key] = xr.open_dataset(file_path)
            except Exception as exc:
                print(f"Falha ao ler {file_path}: {exc}")
    return dataframes, datasets


if os.path.isdir(KAGGLE_INPUT_DIR):
    DATA_PATH = KAGGLE_INPUT_DIR
    print(f"Rodando no Kaggle, dados em {DATA_PATH}")
else:
    import kagglehub
    DATA_PATH = kagglehub.competition_download(COMPETITION)
    print(f"Rodando fora do Kaggle, dados via kagglehub em {DATA_PATH}")

_dataframes, datasets = load_dataset_files(DATA_PATH)
print(f"{len(_dataframes)} csv + {len(datasets)} netcdf carregados: {sorted(set(_dataframes) | set(datasets))}")

In [ ]:
FEATURE_VARS = [
    "t2", "cloud_cover", "shum_850", "surface_pressure", "u_850",
    "v_850", "temperature_850", "rel_hum_850", "geopotential_850",
]
TP_VAR = "tp"
ALL_VARS = FEATURE_VARS + [TP_VAR]
TRAIN_FILES = {
    "t2": "treino_t2", "cloud_cover": "treino_cloud_cover", "shum_850": "treino_shum_850",
    "surface_pressure": "treino_surface_pressure", "u_850": "treino_u_850", "v_850": "treino_v_850",
    "temperature_850": "treino_temperature_850", "rel_hum_850": "treino_rel_hum_850",
    "geopotential_850": "treino_geopotential_850", "tp": "treino_tp",
}
TRAIN_END = "2018-12-01"  # limite de ajuste da reducao (mesmo do run promovido)
HINDCAST_LEN = 12
LAGS = range(1, 25)


def stack_features(datasets):
    data_vars = {var: datasets[key][var] for var, key in TRAIN_FILES.items()}
    return xr.Dataset(data_vars)


def compute_normalization_stats(ds, train_end=TRAIN_END):
    ds_train = ds.sel(time=slice(None, train_end))
    return {
        var: (float(ds_train[var].mean(skipna=True, dtype="float64")), float(ds_train[var].std(skipna=True, dtype="float64")))
        for var in ds.data_vars
    }


def build_examples(component_series, origin_start_idx, origin_end_idx, last_valid_idx,
                    hindcast_len=HINDCAST_LEN, lags=LAGS, oni_series=None):
    """Gera exemplos (hindcast, alvo_atm, tp_congelado, lag) -> y. Contrato completo
    (anti-vazamento temporal) documentado em src/data.py do repositorio."""
    atm_vars = [v for v in ALL_VARS if v != TP_VAR]
    hindcast_list, alvo_atm_list, tp_congelado_list, lag_list, y_list = [], [], [], [], []
    origin_idx_list, alvo_idx_list, oni_list = [], [], []
    for o in range(max(origin_start_idx, hindcast_len - 1), origin_end_idx + 1):
        hindcast_o = np.concatenate([component_series[v][o - hindcast_len + 1 : o + 1] for v in ALL_VARS], axis=1)
        tp_congelado_o = component_series[TP_VAR][o]
        for lag in lags:
            alvo_idx = o + lag
            if alvo_idx > last_valid_idx:
                break
            feature_idx = alvo_idx - 1
            if oni_series is not None:
                oni_valor = oni_series[feature_idx]
                if np.isnan(oni_valor):
                    continue
            alvo_atm = np.concatenate([component_series[v][feature_idx] for v in atm_vars])
            y = component_series[TP_VAR][alvo_idx]
            hindcast_list.append(hindcast_o)
            alvo_atm_list.append(alvo_atm)
            tp_congelado_list.append(tp_congelado_o)
            lag_list.append(lag / max(lags))
            y_list.append(y)
            origin_idx_list.append(o)
            alvo_idx_list.append(alvo_idx)
            if oni_series is not None:
                oni_list.append([oni_valor])
    resultado = (
        np.asarray(hindcast_list, dtype=np.float32),
        np.asarray(alvo_atm_list, dtype=np.float32),
        np.asarray(tp_congelado_list, dtype=np.float32),
        np.asarray(lag_list, dtype=np.float32),
        np.asarray(y_list, dtype=np.float32),
        np.asarray(origin_idx_list, dtype=np.int64),
        np.asarray(alvo_idx_list, dtype=np.int64),
    )
    if oni_series is not None:
        resultado = resultado + (np.asarray(oni_list, dtype=np.float32),)
    return resultado


ds = stack_features(datasets)
teste_ds = datasets["teste_features"]
time_index = pd.DatetimeIndex(ds["time"].values)
last_valid_idx = len(time_index) - 1
train_end_idx = time_index.get_indexer([pd.Timestamp(TRAIN_END)])[0]
stats = compute_normalization_stats(ds, train_end=TRAIN_END)
print(f"{len(time_index)} meses de historico (1940-2022); reducao ajustada ate indice {train_end_idx} ({TRAIN_END})")

## 2. Reducao dimensional (PLS defasado)

Pode levar alguns minutos - e a etapa mais cara do notebook (NIPALS do PLS para 9
variaveis atmosfericas, mais o PCA de `tp`).

In [ ]:
VARIANCE_THRESHOLD = 0.90
PCA_MAX_COMPONENTS = 40
PLS_MAX_COMPONENTS = 15
PLS_MAX_ITER = 100
N_JOBS_REDUCTION = 2  # cauteloso com memoria: cada worker guarda sua propria copia da grade (~313MB/variavel)
PLS_LAG_SHIFT = 1


class SpatialPCA:
    """PCA por variavel sobre a dimensao espacial (lat*lon), ajustado so no treino.
    Numero de componentes = o menor que atinge `variance_threshold`, ate um teto."""

    def __init__(self, variance_threshold=0.90, max_components=200, random_state=42):
        self.variance_threshold = variance_threshold
        self.max_components = max_components
        self.random_state = random_state
        self._pca = None
        self.n_components_ = None
        self.spatial_shape = None

    def fit(self, data):
        self.spatial_shape = data.shape[1:]
        flat = data.reshape(data.shape[0], -1)
        cap = min(self.max_components, flat.shape[0] - 1, flat.shape[1])
        probe = PCA(n_components=cap, svd_solver="randomized", random_state=self.random_state)
        probe.fit(flat)
        cumulative = np.cumsum(probe.explained_variance_ratio_)
        if cumulative[-1] < self.variance_threshold:
            self.n_components_ = cap
            self._pca = probe
        else:
            self.n_components_ = int(np.searchsorted(cumulative, self.variance_threshold)) + 1
            self._pca = PCA(n_components=self.n_components_, svd_solver="randomized", random_state=self.random_state)
            self._pca.fit(flat)
        return self

    def transform(self, data):
        flat = data.reshape(data.shape[0], -1)
        return self._pca.transform(flat)

    def inverse_transform(self, coeffs):
        flat = self._pca.inverse_transform(coeffs)
        return flat.reshape(coeffs.shape[0], *self.spatial_shape)

    def explained_variance_ratio(self):
        return float(self._pca.explained_variance_ratio_.sum())


class SpatialPLS:
    """PLS supervisionado (por Y = componentes de `tp`) por variavel, sobre lat*lon.
    Numero de componentes escolhido por busca binaria (a fracao de variancia de X
    capturada e nao-decrescente em k)."""

    def __init__(self, variance_threshold=0.90, max_components=30, max_iter=100):
        self.variance_threshold = variance_threshold
        self.max_components = max_components
        self.max_iter = max_iter
        self._pls = None
        self.n_components_ = None
        self.spatial_shape = None
        self._x_total_var = None

    def _fit_at(self, flat, target, k):
        pls = PLSRegression(n_components=k, scale=False, max_iter=self.max_iter)
        pls.fit(flat, target)
        ratio = float(np.var(pls.x_scores_, axis=0).sum()) / self._x_total_var
        return pls, ratio

    def fit(self, data, target):
        self.spatial_shape = data.shape[1:]
        flat = data.reshape(data.shape[0], -1)
        self._x_total_var = float(np.var(flat, axis=0).sum())
        cap = min(self.max_components, flat.shape[0] - 1, flat.shape[1])
        best_pls, best_ratio, best_k = *self._fit_at(flat, target, cap), cap
        if best_ratio >= self.variance_threshold:
            lo, hi = 1, cap
            while lo < hi:
                mid = (lo + hi) // 2
                pls, ratio = self._fit_at(flat, target, mid)
                if ratio >= self.variance_threshold:
                    hi = mid
                    best_pls, best_ratio, best_k = pls, ratio, mid
                else:
                    lo = mid + 1
        self._pls = best_pls
        self.n_components_ = best_k
        return self

    def transform(self, data):
        flat = data.reshape(data.shape[0], -1)
        return self._pls.transform(flat)

    def inverse_transform(self, coeffs):
        flat = self._pls.inverse_transform(coeffs)
        return flat.reshape(coeffs.shape[0], *self.spatial_shape)

    def explained_variance_ratio(self):
        scores_var = float(np.var(self._pls.x_scores_, axis=0).sum())
        return scores_var / self._x_total_var


def _fit_one_atm_variable(var, normalizado, method, train_end_idx, pls_lag_shift, tp_components_full):
    if method == "pca":
        reducer = SpatialPCA(variance_threshold=VARIANCE_THRESHOLD, max_components=PCA_MAX_COMPONENTS)
        reducer.fit(normalizado[: train_end_idx + 1])
    else:
        shift = pls_lag_shift if method == "pls_lagged" else 0
        x_fit = normalizado[: train_end_idx + 1 - shift]
        y_fit = tp_components_full[shift : train_end_idx + 1]
        reducer = SpatialPLS(variance_threshold=VARIANCE_THRESHOLD, max_components=PLS_MAX_COMPONENTS, max_iter=PLS_MAX_ITER)
        reducer.fit(x_fit, y_fit)
    componentes = reducer.transform(normalizado).astype("float32")
    log_msg = (
        f"  {method.upper()}[{var}]: {reducer.n_components_} componentes explicam "
        f"{reducer.explained_variance_ratio():.1%} da variancia de X"
    )
    return var, reducer, componentes, log_msg


def fit_reduction_per_variable(ds, stats, train_end_idx, method="pls_lagged", pls_lag_shift=PLS_LAG_SHIFT, n_jobs=N_JOBS_REDUCTION):
    reduction_objects, component_series = {}, {}
    media_tp, desvio_tp = stats[TP_VAR]
    bruto_tp = ds[TP_VAR].values.astype("float32")
    normalizado_tp = (bruto_tp - media_tp) / desvio_tp
    del bruto_tp
    pca_tp = SpatialPCA(variance_threshold=VARIANCE_THRESHOLD, max_components=PCA_MAX_COMPONENTS)
    pca_tp.fit(normalizado_tp[: train_end_idx + 1])
    print(f"  PCA[{TP_VAR}] (referencia): {pca_tp.n_components_} componentes explicam {pca_tp.explained_variance_ratio():.1%} da variancia")
    tp_components_full = pca_tp.transform(normalizado_tp).astype("float32")
    del normalizado_tp
    reduction_objects[TP_VAR] = pca_tp
    component_series[TP_VAR] = tp_components_full

    def _tarefas():
        for var in FEATURE_VARS:
            media, desvio = stats[var]
            bruto = ds[var].values.astype("float32")
            normalizado = (bruto - media) / desvio
            del bruto
            yield joblib.delayed(_fit_one_atm_variable)(var, normalizado, method, train_end_idx, pls_lag_shift, tp_components_full)

    resultados = joblib.Parallel(n_jobs=n_jobs, pre_dispatch="n_jobs")(_tarefas())
    for var, reducer, componentes, log_msg in resultados:
        print(log_msg)
        reduction_objects[var] = reducer
        component_series[var] = componentes
    return reduction_objects, component_series


def reconstruct_tp(pca_tp, stats_tp, coeffs_norm_pca):
    media, desvio = stats_tp
    grid_normalizado = pca_tp.inverse_transform(coeffs_norm_pca)
    return grid_normalizado * desvio + media


print("Ajustando a reducao PLS(defasado)...")
reduction_objects, component_series = fit_reduction_per_variable(
    ds, stats, train_end_idx, method="pls_lagged", pls_lag_shift=PLS_LAG_SHIFT, n_jobs=N_JOBS_REDUCTION
)
tp_pca, stats_tp = reduction_objects[TP_VAR], stats[TP_VAR]
n_tp = tp_pca.n_components_
n_atm = sum(reduction_objects[v].n_components_ for v in FEATURE_VARS)
n_hind = n_atm + n_tp
print(f"n_tp={n_tp}, n_atm={n_atm}, n_hind={n_hind}")

## 3. Indice ONI (El Nino/La Nina)

Usado como feature extra por um dos 3 membros do blend - 2023-2024 (o teste real)
cobre um dos El Ninos mais fortes do registro (ONI chegou a +2.0 em NDJ/2023).
Fonte: NOAA CPC, ONI v6.

In [ ]:
# cada linha e um ano, 12 valores na ordem DJF JFM FMA MAM AMJ MJJ JJA JAS ASO SON OND NDJ
_ONI_TABLE = {
    1950: [-1.3, -1.2, -1.1, -1.1, -1.1, -0.9, -0.6, -0.6, -0.6, -0.6, -0.7, -0.8],
    1951: [-0.7, -0.4, 0.0, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0, 1.0, 1.0, 0.7],
    1952: [0.5, 0.3, 0.3, 0.2, -0.1, -0.4, -0.4, -0.2, 0.0, 0.0, -0.1, 0.1],
    1953: [0.3, 0.5, 0.5, 0.6, 0.8, 0.7, 0.6, 0.7, 0.6, 0.8, 0.5, 0.6],
    1954: [0.4, 0.4, 0.0, -0.3, -0.5, -0.5, -0.7, -0.8, -0.7, -0.5, -0.4, -0.4],
    1955: [-0.5, -0.5, -0.6, -0.8, -0.8, -0.9, -0.7, -0.9, -1.1, -1.4, -1.6, -1.5],
    1956: [-1.2, -0.8, -0.6, -0.5, -0.5, -0.5, -0.5, -0.5, -0.4, -0.4, -0.4, -0.4],
    1957: [-0.3, -0.1, 0.3, 0.6, 0.7, 0.8, 1.0, 1.0, 1.1, 1.2, 1.4, 1.6],
    1958: [1.7, 1.6, 1.2, 0.8, 0.6, 0.5, 0.4, 0.3, 0.3, 0.3, 0.4, 0.6],
    1959: [0.6, 0.6, 0.5, 0.4, 0.2, 0.0, -0.2, -0.2, 0.0, 0.1, 0.2, 0.1],
    1960: [0.0, 0.0, 0.1, 0.2, 0.0, 0.0, 0.0, 0.2, 0.2, 0.1, -0.1, -0.1],
    1961: [0.0, 0.0, 0.0, 0.1, 0.2, 0.2, 0.1, -0.2, -0.3, -0.2, -0.1, 0.0],
    1962: [-0.2, -0.2, -0.1, -0.2, -0.2, -0.1, 0.0, -0.1, -0.2, -0.2, -0.2, -0.3],
    1963: [-0.3, -0.2, 0.0, 0.1, 0.1, 0.5, 0.8, 1.1, 1.1, 1.1, 1.2, 1.1],
    1964: [0.9, 0.5, 0.1, -0.3, -0.6, -0.7, -0.7, -0.7, -0.7, -0.8, -0.8, -0.7],
    1965: [-0.5, -0.2, 0.0, 0.2, 0.5, 0.8, 1.1, 1.3, 1.6, 1.7, 1.8, 1.6],
    1966: [1.2, 1.0, 0.8, 0.4, 0.2, 0.2, 0.3, 0.1, -0.1, -0.1, -0.2, -0.3],
    1967: [-0.5, -0.5, -0.5, -0.4, -0.1, 0.1, 0.0, -0.3, -0.4, -0.4, -0.4, -0.4],
    1968: [-0.6, -0.7, -0.6, -0.5, -0.1, 0.2, 0.4, 0.4, 0.3, 0.5, 0.7, 0.9],
    1969: [1.0, 1.0, 0.8, 0.7, 0.6, 0.5, 0.4, 0.4, 0.6, 0.7, 0.8, 0.7],
    1970: [0.5, 0.2, 0.2, 0.3, 0.1, -0.3, -0.5, -0.7, -0.7, -0.7, -0.8, -1.1],
    1971: [-1.3, -1.4, -1.1, -0.8, -0.7, -0.7, -0.7, -0.6, -0.7, -0.8, -0.9, -0.8],
    1972: [-0.6, -0.3, 0.0, 0.3, 0.5, 0.7, 1.0, 1.3, 1.5, 1.7, 1.8, 1.8],
    1973: [1.6, 1.1, 0.6, 0.0, -0.5, -0.8, -1.0, -1.1, -1.3, -1.6, -1.9, -2.0],
    1974: [-1.9, -1.6, -1.2, -1.0, -0.7, -0.6, -0.5, -0.4, -0.5, -0.7, -0.9, -0.7],
    1975: [-0.6, -0.5, -0.6, -0.7, -0.8, -0.9, -1.1, -1.1, -1.3, -1.3, -1.5, -1.6],
    1976: [-1.5, -1.1, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 0.8, 0.8],
    1977: [0.7, 0.7, 0.3, 0.3, 0.3, 0.5, 0.4, 0.4, 0.5, 0.7, 0.8, 0.8],
    1978: [0.7, 0.5, 0.2, -0.1, -0.3, -0.2, -0.4, -0.4, -0.4, -0.3, -0.1, 0.1],
    1979: [0.2, 0.2, 0.3, 0.3, 0.3, 0.1, 0.1, 0.2, 0.3, 0.5, 0.5, 0.6],
    1980: [0.6, 0.5, 0.4, 0.3, 0.5, 0.4, 0.2, -0.1, -0.2, -0.1, 0.1, 0.0],
    1981: [-0.3, -0.4, -0.4, -0.3, -0.3, -0.3, -0.3, -0.2, -0.1, -0.2, -0.2, -0.1],
    1982: [0.0, 0.0, 0.1, 0.3, 0.7, 0.7, 0.8, 1.0, 1.5, 1.8, 2.0, 2.1],
    1983: [2.1, 1.9, 1.5, 1.2, 1.0, 0.6, 0.3, -0.1, -0.4, -0.8, -1.0, -0.9],
    1984: [-0.6, -0.5, -0.4, -0.5, -0.5, -0.5, -0.4, -0.2, -0.3, -0.6, -0.9, -1.1],
    1985: [-1.0, -0.9, -0.8, -0.8, -0.8, -0.6, -0.4, -0.3, -0.3, -0.3, -0.3, -0.4],
    1986: [-0.4, -0.4, -0.2, -0.2, -0.1, 0.0, 0.2, 0.5, 0.6, 0.9, 1.1, 1.2],
    1987: [1.2, 1.1, 1.0, 0.9, 1.0, 1.1, 1.3, 1.5, 1.5, 1.4, 1.2, 1.0],
    1988: [0.6, 0.4, 0.0, -0.4, -1.0, -1.3, -1.3, -1.1, -1.3, -1.5, -1.8, -1.8],
    1989: [-1.6, -1.4, -1.1, -0.8, -0.6, -0.4, -0.4, -0.4, -0.3, -0.2, -0.1, 0.0],
    1990: [0.2, 0.2, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3],
    1991: [0.4, 0.3, 0.2, 0.3, 0.5, 0.6, 0.8, 0.7, 0.7, 0.8, 1.2, 1.4],
    1992: [1.5, 1.5, 1.4, 1.3, 1.1, 0.7, 0.4, 0.1, -0.1, -0.2, -0.2, 0.0],
    1993: [0.1, 0.3, 0.5, 0.7, 0.8, 0.6, 0.3, 0.2, 0.2, 0.2, 0.2, 0.2],
    1994: [0.1, 0.0, 0.0, 0.2, 0.3, 0.2, 0.3, 0.3, 0.5, 0.7, 0.9, 1.0],
    1995: [0.9, 0.7, 0.5, 0.2, 0.0, -0.1, -0.3, -0.5, -0.8, -1.0, -1.0, -1.0],
    1996: [-0.9, -0.7, -0.5, -0.4, -0.2, -0.2, -0.2, -0.3, -0.3, -0.3, -0.4, -0.4],
    1997: [-0.4, -0.3, 0.0, 0.3, 0.7, 1.1, 1.5, 1.8, 2.0, 2.2, 2.3, 2.4],
    1998: [2.2, 1.9, 1.4, 1.0, 0.4, -0.1, -0.6, -0.8, -1.0, -1.1, -1.3, -1.5],
    1999: [-1.5, -1.2, -0.9, -0.8, -0.8, -0.8, -0.9, -0.9, -1.0, -1.1, -1.3, -1.5],
    2000: [-1.5, -1.3, -1.0, -0.8, -0.7, -0.6, -0.5, -0.4, -0.4, -0.5, -0.8, -0.8],
    2001: [-0.8, -0.5, -0.4, -0.3, -0.2, -0.1, 0.0, -0.1, -0.1, -0.3, -0.3, -0.3],
    2002: [-0.1, 0.1, 0.1, 0.2, 0.3, 0.5, 0.6, 0.6, 0.8, 1.0, 1.2, 1.1],
    2003: [0.9, 0.6, 0.4, 0.0, -0.2, -0.1, 0.1, 0.1, 0.2, 0.2, 0.3, 0.3],
    2004: [0.3, 0.3, 0.2, 0.1, 0.1, 0.2, 0.4, 0.5, 0.6, 0.6, 0.6, 0.7],
    2005: [0.6, 0.6, 0.4, 0.4, 0.3, 0.1, 0.0, -0.1, -0.1, -0.3, -0.6, -0.8],
    2006: [-0.9, -0.8, -0.6, -0.4, -0.2, 0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.9],
    2007: [0.6, 0.2, -0.1, -0.3, -0.3, -0.4, -0.5, -0.8, -1.0, -1.4, -1.6, -1.7],
    2008: [-1.8, -1.6, -1.4, -1.0, -0.8, -0.6, -0.3, -0.2, -0.2, -0.4, -0.6, -0.8],
    2009: [-0.9, -0.8, -0.6, -0.3, 0.0, 0.3, 0.5, 0.6, 0.7, 0.9, 1.3, 1.5],
    2010: [1.5, 1.2, 0.9, 0.4, -0.2, -0.7, -1.0, -1.2, -1.4, -1.5, -1.6, -1.5],
    2011: [-1.3, -1.1, -0.8, -0.7, -0.5, -0.3, -0.4, -0.5, -0.8, -0.9, -1.0, -0.9],
    2012: [-0.7, -0.6, -0.4, -0.3, -0.2, 0.0, 0.3, 0.4, 0.4, 0.3, 0.1, 0.0],
    2013: [-0.2, -0.3, -0.2, -0.2, -0.3, -0.4, -0.4, -0.3, -0.3, -0.1, -0.1, -0.1],
    2014: [-0.2, -0.2, 0.0, 0.3, 0.4, 0.2, 0.1, 0.1, 0.2, 0.5, 0.7, 0.8],
    2015: [0.7, 0.7, 0.7, 0.9, 1.0, 1.2, 1.4, 1.7, 2.0, 2.3, 2.5, 2.6],
    2016: [2.5, 2.2, 1.7, 1.1, 0.6, 0.1, -0.2, -0.3, -0.4, -0.5, -0.5, -0.4],
    2017: [-0.1, 0.1, 0.3, 0.3, 0.4, 0.3, 0.1, -0.1, -0.2, -0.4, -0.6, -0.8],
    2018: [-0.7, -0.7, -0.5, -0.3, -0.1, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.1],
    2019: [1.0, 0.9, 0.9, 0.8, 0.7, 0.6, 0.4, 0.2, 0.3, 0.5, 0.7, 0.8],
    2020: [0.7, 0.7, 0.6, 0.3, 0.0, -0.2, -0.3, -0.4, -0.8, -1.0, -1.1, -1.1],
    2021: [-1.0, -0.9, -0.7, -0.6, -0.4, -0.3, -0.3, -0.5, -0.6, -0.8, -0.9, -0.8],
    2022: [-0.8, -0.7, -0.8, -0.9, -0.8, -0.7, -0.7, -0.8, -0.9, -0.9, -0.8, -0.7],
    2023: [-0.5, -0.3, -0.1, 0.2, 0.5, 0.7, 1.0, 1.3, 1.5, 1.7, 1.9, 2.0],
    2024: [1.8, 1.5, 1.2, 0.8, 0.4, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4],
}
ONI_MONTHLY = {(ano, mes + 1): valor for ano, valores in _ONI_TABLE.items() for mes, valor in enumerate(valores)}


def oni_para_datas(datas):
    """Levanta erro se faltar ONI (uso em validacao/teste, cobertura 1950-2024 garantida)."""
    faltando = [d for d in datas if (d.year, d.month) not in ONI_MONTHLY]
    if faltando:
        raise ValueError(f"ONI nao disponivel para: {faltando[:5]}...")
    return np.array([ONI_MONTHLY[(d.year, d.month)] for d in datas], dtype="float32")


def oni_series_or_nan(datas):
    """Como oni_para_datas, mas NaN (nao erro) para datas fora da tabela - origens de
    treino em 1940-1949 nao tem ONI disponivel."""
    return np.array([ONI_MONTHLY.get((d.year, d.month), np.nan) for d in datas], dtype="float32")

## 4. LSTM hindcast/forecast

In [ ]:
class HindcastForecastLSTM(nn.Module):
    """Encoder LSTM sobre a janela historica + decoder condicionado no ultimo mes com
    dado atmosferico disponivel antes do alvo, no tp congelado e no lag."""

    def __init__(self, n_features_hindcast, n_features_atm, n_components_tp,
                 hidden_size=128, num_layers=1, dropout=0.1, n_oni_features=0):
        super().__init__()
        self.n_oni_features = n_oni_features
        self.encoder = nn.LSTM(
            input_size=n_features_hindcast, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        decoder_input_dim = hidden_size + n_features_atm + n_components_tp + 1 + n_oni_features
        self.decoder = nn.Sequential(
            nn.Linear(decoder_input_dim, hidden_size), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, n_components_tp),
        )

    def forward(self, hindcast_seq, target_month_features, tp_frozen, lag, oni=None):
        _, (hn, _) = self.encoder(hindcast_seq)
        context = hn[-1]
        if lag.dim() == 1:
            lag = lag.unsqueeze(-1)
        partes = [context, target_month_features, tp_frozen, lag]
        if self.n_oni_features > 0:
            partes.append(oni)
        decoder_input = torch.cat(partes, dim=-1)
        return self.decoder(decoder_input)

## 5. As 3 configs do blend

Escolhidas (config, numero de epocas, pesos do blend) por uma CV walk-forward de 5
dobras rodada separadamente (`cv_ensemble.py` no repositorio) - aqui so os valores
finais sao usados, nao a busca em si.

In [ ]:
BATCH_SIZE = 64

CONFIGS = {
    "anom_wd_do3": dict(hidden_size=128, dropout=0.3, lr=5e-4, weight_decay=0.5, anomaly=True),
    "anom_wd_do3_oni": dict(hidden_size=128, dropout=0.3, lr=5e-4, weight_decay=0.5, anomaly=True, use_oni_feature=True),
    "anom_wd": dict(hidden_size=128, dropout=0.1, lr=5e-4, weight_decay=0.5, anomaly=True),
}
EPOCAS = {"anom_wd_do3": 2, "anom_wd_do3_oni": 3, "anom_wd": 2}
SEEDS = {"anom_wd_do3": 0, "anom_wd_do3_oni": 0, "anom_wd": 0}
ALFAS = {  # pesos do blend (minimos quadrados, CV leave-one-fold-out)
    "anom_wd_do3": 0.032167382538318634,
    "anom_wd_do3_oni": 0.4134542942047119,
    "anom_wd": 0.3644355237483978,
}


def make_loader(hindcast, alvo_atm, tp_congelado, lag, y, batch_size, shuffle, oni=None):
    tensors = [torch.from_numpy(hindcast), torch.from_numpy(alvo_atm), torch.from_numpy(tp_congelado),
               torch.from_numpy(lag), torch.from_numpy(y)]
    if oni is not None:
        tensors.append(torch.from_numpy(oni))
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)


def _predict(model, hindcast, atm, tp_froz, lag, oni=None, batch=512):
    model.eval()
    saidas = []
    with torch.no_grad():
        for i in range(0, len(lag), batch):
            saidas.append(model(
                torch.from_numpy(hindcast[i : i + batch]), torch.from_numpy(atm[i : i + batch]),
                torch.from_numpy(tp_froz[i : i + batch]), torch.from_numpy(lag[i : i + batch]),
                oni=torch.from_numpy(oni[i : i + batch]) if oni is not None else None,
            ).numpy())
    return np.concatenate(saidas)


def train_lstm(cfg, seed, n_hind, n_atm, n_tp, tr, y_tr_target, va, shift_va, epochs, oni_tr=None, oni_va=None):
    """Treina `epochs` epocas fixas (sem early stopping - o numero certo ja foi
    escolhido na CV) e devolve a previsao final (ja somada a `shift_va`)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    hindcast_tr, atm_tr, tp_froz_tr, lag_tr = tr
    n_oni_features = 1 if oni_tr is not None else 0
    loader = make_loader(hindcast_tr, atm_tr, tp_froz_tr, lag_tr, y_tr_target, BATCH_SIZE, shuffle=True, oni=oni_tr)
    model = HindcastForecastLSTM(
        n_features_hindcast=n_hind, n_features_atm=n_atm, n_components_tp=n_tp,
        hidden_size=cfg["hidden_size"], dropout=cfg["dropout"], n_oni_features=n_oni_features,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    criterion = torch.nn.MSELoss()
    for epoch in range(1, epochs + 1):
        model.train()
        soma, n = 0.0, 0
        for batch in loader:
            hindcast, atm, tp_froz, lag, y = batch[:5]
            oni = batch[5] if n_oni_features > 0 else None
            optimizer.zero_grad()
            loss = criterion(model(hindcast, atm, tp_froz, lag, oni=oni), y)
            loss.backward()
            optimizer.step()
            soma += loss.item() * hindcast.size(0)
            n += hindcast.size(0)
        print(f"    epoca {epoch}/{epochs} | treino MSE {soma / n:.3f}", flush=True)
    return _predict(model, *va, oni=oni_va) + shift_va

## 6. Climatologia e exemplos de treino (1940-2022)

In [ ]:
hind, atm, tpf, lag, y, _, alvo = build_examples(component_series, 0, last_valid_idx, last_valid_idx=last_valid_idx)
meses = time_index.month.values
tp_c = component_series[TP_VAR]
clim_comp = np.stack([tp_c[meses == m].mean(axis=0) for m in range(1, 13)]).astype("float32")
clim_tr = clim_comp[meses[alvo] - 1]
tr = (hind, atm, tpf, lag)
print(f"{len(y)} exemplos de treino (1940-2022)")

oni_full = oni_series_or_nan(time_index)
hind_o, atm_o, tpf_o, lag_o, y_oni, _, alvo_o, oni_tr = build_examples(
    component_series, 0, last_valid_idx, last_valid_idx=last_valid_idx, oni_series=oni_full
)
tr_oni = (hind_o, atm_o, tpf_o, lag_o)
clim_tr_oni = clim_comp[meses[alvo_o] - 1]
print(f"{len(y_oni)} exemplos com ONI disponivel (descarta origens antes de 1950)")

## 7. Entradas do teste real (2023-2024)

In [ ]:
hindcast_teste = np.concatenate(
    [component_series[v][last_valid_idx - HINDCAST_LEN + 1 : last_valid_idx + 1] for v in ALL_VARS], axis=1
)
n_teste = teste_ds.sizes["time"]
atm_teste = np.concatenate(
    [reduction_objects[v].transform((teste_ds[v].values.astype("float32") - stats[v][0]) / stats[v][1]) for v in FEATURE_VARS],
    axis=1,
).astype("float32")
hind_te = np.repeat(hindcast_teste[None], n_teste, axis=0).astype("float32")
tpf_te = np.repeat(component_series[TP_VAR][last_valid_idx][None], n_teste, axis=0).astype("float32")
lag_te = (teste_ds["lag_meses"].values / max(LAGS)).astype("float32")
clim_te = clim_comp[pd.DatetimeIndex(teste_ds["time"].values).month.values - 1]
te = (hind_te, atm_teste, tpf_te, lag_te)

datas_alvo_teste = pd.DatetimeIndex(teste_ds["time"].values) - pd.DateOffset(months=1)
oni_te = oni_para_datas(datas_alvo_teste).reshape(-1, 1)
print(f"{n_teste} meses de teste real (2023-2024)")

## 8. Treinando os 3 modelos e combinando o blend

In [ ]:
preds = {}
for nome, cfg in CONFIGS.items():
    epochs, seed = EPOCAS[nome], SEEDS[nome]
    com_oni = cfg.get("use_oni_feature", False)
    print(f"\n-- {nome}: seed {seed}, {epochs} epocas" + (" (com ONI)" if com_oni else ""), flush=True)
    if com_oni:
        preds[nome] = train_lstm(cfg, seed, n_hind, n_atm, n_tp, tr_oni, y_oni - clim_tr_oni, te, clim_te, epochs, oni_tr=oni_tr, oni_va=oni_te)
    else:
        preds[nome] = train_lstm(cfg, seed, n_hind, n_atm, n_tp, tr, y - clim_tr, te, clim_te, epochs)

pred_comp = clim_te + sum(ALFAS[nome] * (preds[nome] - clim_te) for nome in CONFIGS)
print("\nBlend calculado.")

## 9. Reconstruindo a grade e gerando a submissao

In [ ]:
def build_submission(predictions, sample_submission_path, output_path):
    """Formata `predictions` (time, lat, lon) para id,tp_mm_day e valida contra o sample_submission."""
    df = predictions.rename("tp_mm_day").to_dataframe().reset_index()
    df["id"] = (
        df["time"].dt.strftime("%Y_%m") + "_" + df["lat"].map("{:.2f}".format) + "_" + df["lon"].map("{:.2f}".format)
    )
    df = df[["id", "tp_mm_day"]]
    sample = pd.read_csv(sample_submission_path)
    faltando = set(sample["id"]) - set(df["id"])
    extras = set(df["id"]) - set(sample["id"])
    if faltando or extras:
        raise ValueError(f"ids nao batem com sample_submission: {len(faltando)} faltando, {len(extras)} extras")
    df = df.set_index("id").loc[sample["id"]].reset_index()
    df.to_csv(output_path, index=False)
    return df


pred_grid = np.clip(reconstruct_tp(tp_pca, stats_tp, pred_comp), 0, None)  # precipitacao nao e negativa
predictions_da = xr.DataArray(
    pred_grid, dims=("time", "lat", "lon"),
    coords={"time": teste_ds["time"].values, "lat": teste_ds["lat"].values, "lon": teste_ds["lon"].values},
)
sample_path = os.path.join(DATA_PATH, "sample_submission.csv")
output_path = "/kaggle/working/submission.csv" if os.path.isdir("/kaggle/input") else "submission.csv"
df = build_submission(predictions_da, sample_path, output_path)
print(f"Submissao salva em {output_path} ({len(df)} linhas)")
df.head()

## Notas

- RMSE-CV (leave-one-fold-out, 5 dobras walk-forward): **1.778** mm/dia, vs 1.828 da
  climatologia. RMSE real no Kaggle: **1.73116**.
- A CV que escolheu esta configuracao/pesos (`cv_ensemble.py` + `final_blend.py`), o
  sweep de hiperparametros completo e um notebook de resultados mais detalhado
  (`resultados_pca_lstm.ipynb`) estao no repositorio do projeto.
- Nao aplica a correcao de vies ENSO pos-hoc (testada separadamente na CV, piorou o
  RMSE - ver `postprocess_enso.py`).
- Este notebook nao usa nenhum cache de reducao (ajusta o PLS do zero) para ficar
  genuinamente autocontido - rodando de novo com o mesmo `TRAIN_END`/hiperparametros
  reproduz os mesmos componentes (a menos de nao-determinismo do solver do sklearn).